**CNN Attention Sensor-Fusion with Graphics Obstacles**

This version uses a 30x30 white grid and draws tree/apartment/drone/target PNG graphics from `../graphics` inside grid cells. The camera observation is still a local image patch used by the CNN, while LiDAR and IMU remain numeric sensor streams.


In [ ]:
import heapq
import math
import random
import time
from collections import deque
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

In [ ]:
@dataclass
class Config:
    grid_size: int = 30
    start: tuple = (1, 28)
    target: tuple = (28, 1)
    max_steps: int = 180
    episodes: int = 500
    gamma: float = 0.90
    lr: float = 0.0025
    batch_size: int = 64
    memory_size: int = 20_000
    target_update: int = 10
    soft_tau: float = 0.02
    eps_start: float = 0.90
    eps_end: float = 0.02
    eps_decay: float = 450.0
    safe_radius: float = 0.60
    target_reward: float = 500.0
    collision_penalty: float = -300.0
    boundary_penalty: float = -300.0
    distance_scale: float = -0.02
    progress_scale: float = 3.0
    step_penalty: float = -0.10
    grad_clip: float = 10.0
    initial_energy: float = 100.0
    base_energy_cost: float = 0.70
    diagonal_energy_cost: float = 0.30
    corner_energy_cost: float = 0.45
    obstacle_risk_energy_cost: float = 0.25
    camera_warning_energy_cost: float = 0.35
    goal_energy_bonus: float = 25.0
    depleted_energy_penalty: float = -250.0
    energy_reward_scale: float = 0.60
    camera_patch_size: int = 15
    camera_channels: int = 3
    attention_entropy_weight: float = 0.01


cfg = Config()
OUTPUT_DIR = Path('.')
CSV_PATH = OUTPUT_DIR / 'cnn_attention_sensorfusion_graphics_rl_algorithm_metrics.csv'
EPISODE_REWARD_CSV_PATH = OUTPUT_DIR / \
    'cnn_attention_sensorfusion_graphics_rl_episode_rewards.csv'
PLOT_PATH = OUTPUT_DIR / 'cnn_attention_sensorfusion_graphics_rl_algorithm_paths.png'
REWARD_PLOT_PATH = OUTPUT_DIR / \
    'cnn_attention_sensorfusion_graphics_rl_episode_reward_moving_average.png'
BAR_PATH = OUTPUT_DIR / 'cnn_attention_sensorfusion_graphics_rl_algorithm_bar_comparison.png'
ATTENTION_PLOT_PATH = OUTPUT_DIR / \
    'cnn_attention_sensorfusion_graphics_rl_attention_weights.png'
CAMERA_EXAMPLE_PATH = OUTPUT_DIR / \
    'cnn_attention_sensorfusion_graphics_camera_observation.png'
CHECKPOINT_DIR = OUTPUT_DIR / 'cnn_attention_sensorfusion_graphics_rl_checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)
GRAPHICS_DIR = Path('..') / 'graphics'
TREE_IMAGE_PATH = GRAPHICS_DIR / 'tree.png'
APARTMENT_IMAGE_PATH = GRAPHICS_DIR / 'apartments.png'
DRONE_IMAGE_PATH = GRAPHICS_DIR / 'drone.png'
TARGET_IMAGE_PATH = GRAPHICS_DIR / 'target.png'


In [ ]:
def add_rect(cells, x0, x1, y0, y1):
    for x in range(x0, x1 + 1):
        for y in range(y0, y1 + 1):
            cells.add((x, y))


def build_typed_obstacle_map():
    # Icon-style map: each occupied cell is drawn as one tree or apartment.
    # The layout stays sparse enough for A* warm-start and RL exploration.
    tree_cells = {
        (2, 26), (7, 28), (13, 25), (22, 28),
        (5, 22), (9, 20), (13, 19), (20, 22), (23, 22), (26, 20),
        (4, 16), (7, 14), (17, 16), (19, 16), (21, 16), (25, 14),
        (2, 10), (10, 9), (16, 7), (23, 8),
        (6, 5), (12, 4), (20, 4), (26, 5),
    }
    apartment_cells = {
        (15, 28), (18, 25), (4, 24), (17, 22),
        (7, 19), (22, 19), (1, 16), (11, 16), (25, 16),
        (4, 12), (13, 12), (17, 12), (10, 7), (24, 7),
        (15, 2),
    }

    tree_cells -= apartment_cells
    for cells in (tree_cells, apartment_cells):
        cells.discard(cfg.start)
        cells.discard(cfg.target)

    obstacle_types = {cell: 'tree' for cell in tree_cells}
    obstacle_types.update({cell: 'apartment' for cell in apartment_cells})
    obstacles = set(obstacle_types)
    return obstacles, tree_cells, apartment_cells, obstacle_types


OBSTACLES, TREE_CELLS, APARTMENT_CELLS, OBSTACLE_TYPES = build_typed_obstacle_map()
len(OBSTACLES), len(TREE_CELLS), len(APARTMENT_CELLS)


In [ ]:
class CNNCameraSensorFusionUAVGridEnv:
    ACTIONS = np.array([[0, 1], [0, -1], [-1, 0], [1, 0],
                       [1, 1], [1, -1], [-1, 1], [-1, -1]], dtype=np.float32)
    ACTION_NAMES = ['up', 'down', 'left', 'right',
                    'upper_right', 'lower_right', 'upper_left', 'lower_left']
    LIDAR_DIRECTIONS = np.array(
        [[0, 1], [0, -1], [-1, 0], [1, 0]], dtype=np.int32)

    def __init__(self, config, obstacles):
        self.cfg = config
        self.obstacle_cells = set(obstacles)
        self.obstacles = np.array(sorted(obstacles), dtype=np.float32)
        self.start_pos = np.array(config.start, dtype=np.float32)
        self.target_pos = np.array(config.target, dtype=np.float32)
        self.action_size = len(self.ACTIONS)
        self.vector_size = 5 + 12 + 4
        self.camera_shape = (config.camera_channels,
                             config.camera_patch_size, config.camera_patch_size)
        self.reset()

    def reset(self):
        self.pos = self.start_pos.copy()
        self.steps = 0
        self.done = False
        self.energy = float(self.cfg.initial_energy)
        self.last_action = None
        self.last_motion = np.zeros(2, dtype=np.float32)
        self.last_motion_kind = 'start'
        self.path = [tuple(map(int, self.pos))]
        self.energy_history = [self.energy]
        return self._state()

    def _in_bounds(self, cell):
        return 0 <= cell[0] < self.cfg.grid_size and 0 <= cell[1] < self.cfg.grid_size

    def _is_blocked_cell(self, cell):
        cell = tuple(map(int, cell))
        return (not self._in_bounds(cell)) or cell in self.obstacle_cells

    def _target_distance(self, pos=None):
        p = self.pos if pos is None else np.array(pos, dtype=np.float32)
        return float(np.linalg.norm(p - self.target_pos))

    def _obstacle_distance(self, pos=None):
        p = self.pos if pos is None else np.array(pos, dtype=np.float32)
        if len(self.obstacles) == 0:
            return math.sqrt(2) * self.cfg.grid_size
        return float(np.min(np.linalg.norm(self.obstacles - p, axis=1)))

    def _lidar_scan(self):
        readings = []
        current = tuple(map(int, self.pos))
        for direction in self.LIDAR_DIRECTIONS:
            for distance in range(1, 4):
                cell = (current[0] + int(direction[0]) * distance,
                        current[1] + int(direction[1]) * distance)
                readings.append(1.0 if self._is_blocked_cell(cell) else 0.0)
        return np.array(readings, dtype=np.float32)

    def _imu_features(self):
        dx, dy = self.last_motion
        straight = 1.0 if self.last_motion_kind == 'straight' else 0.0
        corner = 1.0 if self.last_motion_kind == 'corner' else 0.0
        return np.array([dx, dy, straight, corner], dtype=np.float32)

    def _camera_image(self):
        size = self.cfg.camera_patch_size
        half = size // 2
        current = tuple(map(int, self.pos))
        img = np.zeros((3, size, size), dtype=np.float32)
        for ix in range(size):
            for iy in range(size):
                world = (current[0] + ix - half, current[1] + iy - half)
                if not self._in_bounds(world):
                    img[:, ix, iy] = np.array(
                        [0.45, 0.45, 0.45], dtype=np.float32)
                elif world in self.obstacle_cells:
                    img[:, ix, iy] = np.array(
                        [0.0, 0.0, 0.0], dtype=np.float32)
                else:
                    img[:, ix, iy] = np.array(
                        [1.0, 1.0, 1.0], dtype=np.float32)
        tx, ty = int(self.target_pos[0]) - current[0] + \
            half, int(self.target_pos[1]) - current[1] + half
        if 0 <= tx < size and 0 <= ty < size:
            img[:, tx, ty] = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        img[:, half, half] = np.array([0.1, 0.35, 1.0], dtype=np.float32)
        return img

    def _vector_state(self):
        max_dist = math.sqrt(2) * self.cfg.grid_size
        od = min(self._obstacle_distance(), max_dist)
        base = np.array([self.pos[0] / self.cfg.grid_size, self.pos[1] / self.cfg.grid_size, self._target_distance() /
                        max_dist, od / max_dist, self.energy / self.cfg.initial_energy], dtype=np.float32)
        return np.concatenate([base, self._lidar_scan(), self._imu_features()]).astype(np.float32)

    def _state(self):
        return {'image': self._camera_image(), 'vector': self._vector_state()}

    def _motion_kind(self, action):
        if self.last_action is None:
            return 'straight'
        previous = self.ACTIONS[self.last_action]
        current = self.ACTIONS[action]
        return 'straight' if np.array_equal(previous, current) else 'corner'

    def _camera_predicts_blocked(self, action):
        current = tuple(map(int, self.pos))
        dx, dy = map(int, self.ACTIONS[action])
        proposed = (current[0] + dx, current[1] + dy)
        blocked = self._is_blocked_cell(proposed)
        if dx != 0 and dy != 0:
            blocked = blocked or self._is_blocked_cell(
                (current[0] + dx, current[1])) or self._is_blocked_cell((current[0], current[1] + dy))
        return blocked

    def _energy_cost(self, action, camera_blocked):
        move = self.ACTIONS[action]
        cost = self.cfg.base_energy_cost
        if abs(move[0]) + abs(move[1]) == 2:
            cost += self.cfg.diagonal_energy_cost
        if self._motion_kind(action) == 'corner':
            cost += self.cfg.corner_energy_cost
        nearby_obstacles = self._lidar_scan().reshape(4, 3)[:, 0].sum()
        cost += self.cfg.obstacle_risk_energy_cost * nearby_obstacles
        if camera_blocked:
            cost += self.cfg.camera_warning_energy_cost
        return float(cost)

    def step(self, action):
        if self.done:
            return self._state(), 0.0, True, {}
        action = int(action)
        previous_td = self._target_distance()
        previous_energy = self.energy
        proposed = self.pos + self.ACTIONS[action]
        camera_blocked = self._camera_predicts_blocked(action)
        energy_cost = self._energy_cost(action, camera_blocked)
        motion_kind = self._motion_kind(action)
        self.steps += 1
        out = proposed[0] < 0 or proposed[0] >= self.cfg.grid_size or proposed[1] < 0 or proposed[1] >= self.cfg.grid_size
        self.pos = proposed
        self.path.append(tuple(map(int, self.pos)))
        td = self._target_distance()
        od = self._obstacle_distance()
        reached = td <= math.sqrt(2) / 2
        collision = tuple(map(int, self.pos)
                          ) in self.obstacle_cells or od < self.cfg.safe_radius
        self.energy = max(0.0, self.energy - energy_cost)
        if reached:
            self.energy = min(self.cfg.initial_energy,
                              self.energy + self.cfg.goal_energy_bonus)
        energy_delta = self.energy - previous_energy
        energy_depleted = self.energy <= 0.0 and not reached
        reward = self.cfg.step_penalty + self.cfg.distance_scale * \
            td + self.cfg.progress_scale * (previous_td - td)
        reward += self.cfg.energy_reward_scale * energy_delta
        if reached:
            reward += self.cfg.target_reward
        if collision:
            reward += self.cfg.collision_penalty
        if out:
            reward += self.cfg.boundary_penalty
        if energy_depleted:
            reward += self.cfg.depleted_energy_penalty
        self.last_action = action
        self.last_motion = self.ACTIONS[action].copy()
        self.last_motion_kind = motion_kind
        self.energy_history.append(self.energy)
        self.done = bool(
            reached or collision or out or energy_depleted or self.steps >= self.cfg.max_steps)
        info = {'target_distance': td, 'obstacle_distance': od, 'reached': reached, 'collision': collision, 'out_of_bounds': out, 'energy': self.energy, 'energy_cost': energy_cost,
                'energy_depleted': energy_depleted, 'motion_kind': motion_kind, 'camera_blocked': camera_blocked, 'lidar_near_obstacles': int(self._lidar_scan().reshape(4, 3)[:, 0].sum())}
        return self._state(), float(reward), self.done, info



In [ ]:
env = CNNCameraSensorFusionUAVGridEnv(cfg, OBSTACLES)
sample_state = env.reset()
plt.figure(figsize=(3, 3))
plt.imshow(np.transpose(sample_state['image'], (1, 2, 0)), origin='lower')
plt.title('CNN camera observation at start')
plt.axis('off')
plt.tight_layout()
plt.savefig(CAMERA_EXAMPLE_PATH, dpi=180)
plt.show()
env.vector_size, env.camera_shape, env.action_size

In [ ]:

class CameraCNN(nn.Module):
    def __init__(self, in_channels=3, patch_size=15, embed_dim=64):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(
        ), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        pooled = patch_size // 4
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(
            32 * pooled * pooled, embed_dim), nn.ReLU())

    def forward(self, image):
        return self.proj(self.features(image))


class CNNQNetwork(nn.Module):
    def __init__(self, vector_size, action_size, camera_shape, embed_dim=64):
        super().__init__()
        c, h, _ = camera_shape
        self.camera = CameraCNN(c, h, embed_dim)
        self.vector = nn.Sequential(
            nn.Linear(vector_size, embed_dim), nn.ReLU())
        self.net = nn.Sequential(nn.Linear(embed_dim * 2, 128), nn.ReLU(), nn.Linear(
            128, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, action_size))

    def forward(self, image, vector):
        z = torch.cat([self.camera(image), self.vector(vector)], dim=1)
        return self.net(z)


class CNNDuelingQNetwork(nn.Module):
    def __init__(self, vector_size, action_size, camera_shape, embed_dim=64):
        super().__init__()
        c, h, _ = camera_shape
        self.camera = CameraCNN(c, h, embed_dim)
        self.vector = nn.Sequential(
            nn.Linear(vector_size, embed_dim), nn.ReLU())
        self.shared = nn.Sequential(nn.Linear(embed_dim * 2, 256), nn.ReLU())
        self.value = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))
        self.advantage = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, action_size))

    def forward(self, image, vector):
        z = self.shared(
            torch.cat([self.camera(image), self.vector(vector)], dim=1))
        value = self.value(z)
        advantage = self.advantage(z)
        return value + advantage - advantage.mean(dim=1, keepdim=True)


class GoalConditionedCNNAttentionDuelingQNetwork(nn.Module):
    def __init__(self, vector_size, action_size, camera_shape, embed_dim=64, num_heads=4):
        super().__init__()
        c, h, _ = camera_shape
        self.base_encoder = nn.Sequential(nn.Linear(5, embed_dim), nn.ReLU())
        self.lidar_encoder = nn.Sequential(nn.Linear(12, embed_dim), nn.ReLU())
        self.camera_encoder = CameraCNN(c, h, embed_dim)
        self.imu_encoder = nn.Sequential(nn.Linear(4, embed_dim), nn.ReLU())
        self.token_type_embed = nn.Parameter(torch.zeros(3, embed_dim))
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=num_heads, dropout=0.0, batch_first=True)
        nn.init.zeros_(self.attention.out_proj.weight)
        nn.init.zeros_(self.attention.out_proj.bias)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(
            embed_dim, embed_dim * 2), nn.ReLU(), nn.Linear(embed_dim * 2, embed_dim))
        self.norm2 = nn.LayerNorm(embed_dim)
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, 256), nn.ReLU(), nn.Linear(256, embed_dim))
        self.shared = nn.Sequential(nn.Linear(embed_dim, 256), nn.ReLU())
        self.value = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))
        self.advantage = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, action_size))
        self.last_attention = None

    def forward(self, image, vector):
        base = vector[:, 0:5]
        lidar = vector[:, 5:17]
        imu = vector[:, 17:21]
        base_feat = self.base_encoder(base)
        sensor_tokens = torch.stack([self.lidar_encoder(
            lidar), self.camera_encoder(image), self.imu_encoder(imu)], dim=1)
        sensor_tokens = self.norm_kv(
            sensor_tokens + self.token_type_embed.unsqueeze(0))
        query = self.norm_q(base_feat).unsqueeze(1)
        attn_out, attn_weights = self.attention(
            query, sensor_tokens, sensor_tokens, need_weights=True)
        self.last_attention = attn_weights.detach().cpu()
        x = self.norm1(query + attn_out)
        x = self.norm2(x + self.ffn(x)).squeeze(1)
        fused = self.fusion(torch.cat([base_feat, x], dim=1))
        z = self.shared(fused)
        value = self.value(z)
        advantage = self.advantage(z)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.data = deque(maxlen=capacity)

    def __len__(self):
        return len(self.data)

    def push(self, state, action, reward, next_state, done):
        self.data.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.data, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        images = torch.tensor(
            np.stack([s['image'] for s in states]), dtype=torch.float32, device=DEVICE)
        vectors = torch.tensor(
            np.stack([s['vector'] for s in states]), dtype=torch.float32, device=DEVICE)
        next_images = torch.tensor(np.stack(
            [s['image'] for s in next_states]), dtype=torch.float32, device=DEVICE)
        next_vectors = torch.tensor(np.stack(
            [s['vector'] for s in next_states]), dtype=torch.float32, device=DEVICE)
        return images, vectors, torch.tensor(actions, dtype=torch.long, device=DEVICE).unsqueeze(1), torch.tensor(rewards, dtype=torch.float32, device=DEVICE).unsqueeze(1), next_images, next_vectors, torch.tensor(dones, dtype=torch.float32, device=DEVICE).unsqueeze(1)

In [ ]:
class RLAgent:
    def __init__(self, env, config, algorithm):
        self.cfg = config
        self.algorithm = algorithm
        self.action_size = env.action_size
        if algorithm == 'CNN-A-IDDQN':
            net_cls = GoalConditionedCNNAttentionDuelingQNetwork
        elif algorithm in {'CNN-IDDQN', 'CNN-Dueling DQN'}:
            net_cls = CNNDuelingQNetwork
        else:
            net_cls = CNNQNetwork
        self.current_net = net_cls(
            env.vector_size, env.action_size, env.camera_shape).to(DEVICE)
        self.target_net = net_cls(
            env.vector_size, env.action_size, env.camera_shape).to(DEVICE)
        self.target_net.load_state_dict(self.current_net.state_dict())
        self.target_net.eval()
        self.memory = ReplayBuffer(config.memory_size)
        self.optimizer = optim.Adam(
            self.current_net.parameters(), lr=config.lr)
        self.loss_fn = nn.MSELoss()

    def epsilon(self, episode):
        if self.algorithm in {'CNN-IDDQN', 'CNN-A-IDDQN'}:
            return self.cfg.eps_end + (self.cfg.eps_start - self.cfg.eps_end) / (1.0 + math.exp(episode / self.cfg.eps_decay))
        return max(self.cfg.eps_end, max(0.05, self.cfg.eps_start * (0.995 ** episode)))

    def _state_to_tensors(self, state):
        image = torch.tensor(
            state['image'], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        vector = torch.tensor(
            state['vector'], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        return image, vector

    def act(self, state, episode=10_000, greedy=False):
        eps = 0.0 if greedy else self.epsilon(episode)
        if random.random() < eps:
            return random.randrange(self.action_size)
        was_training = self.current_net.training
        if greedy:
            self.current_net.eval()
        with torch.no_grad():
            image, vector = self._state_to_tensors(state)
            action = int(self.current_net(image, vector).argmax(dim=1).item())
        if greedy and was_training:
            self.current_net.train()
        return action

    def learn(self):
        if len(self.memory) < self.cfg.batch_size:
            return None
        self.current_net.train()
        images, vectors, actions, rewards, next_images, next_vectors, dones = self.memory.sample(
            self.cfg.batch_size)
        q_values = self.current_net(images, vectors).gather(1, actions)
        with torch.no_grad():
            if self.algorithm in {'CNN-Double DQN', 'CNN-IDDQN', 'CNN-A-IDDQN'}:
                next_actions = self.current_net(
                    next_images, next_vectors).argmax(dim=1, keepdim=True)
                next_q = self.target_net(
                    next_images, next_vectors).gather(1, next_actions)
            else:
                next_q = self.target_net(next_images, next_vectors).max(
                    dim=1, keepdim=True).values
            target = rewards + self.cfg.gamma * next_q * (1.0 - dones)
        loss = self.loss_fn(q_values, target)
        if self.algorithm == 'CNN-A-IDDQN' and self.current_net.last_attention is not None and self.cfg.attention_entropy_weight > 0:
            attn = self.current_net.last_attention.to(
                DEVICE).squeeze(1).clamp_min(1e-8)
            entropy = -(attn * attn.log()).sum(dim=1).mean()
            loss = loss + self.cfg.attention_entropy_weight * entropy
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            self.current_net.parameters(), self.cfg.grad_clip)
        self.optimizer.step()
        return float(loss.item())

    def update_target(self, soft=False):
        if soft:
            tau = self.cfg.soft_tau
            for target_param, current_param in zip(self.target_net.parameters(), self.current_net.parameters()):
                target_param.data.copy_(
                    tau * current_param.data + (1.0 - tau) * target_param.data)
        else:
            self.target_net.load_state_dict(self.current_net.state_dict())

In [ ]:
def astar_path(start=cfg.start, target=cfg.target):
    obstacles = set(OBSTACLES)
    actions = [tuple(map(int, a))
               for a in CNNCameraSensorFusionUAVGridEnv.ACTIONS]

    def h(cell):
        return math.hypot(cell[0] - target[0], cell[1] - target[1])
    queue = [(h(start), 0.0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0.0}
    while queue:
        _, cost, current = heapq.heappop(queue)
        if current == target:
            break
        for dx, dy in actions:
            nxt = (current[0] + dx, current[1] + dy)
            if nxt[0] < 0 or nxt[0] >= cfg.grid_size or nxt[1] < 0 or nxt[1] >= cfg.grid_size or nxt in obstacles:
                continue
            new_cost = cost + math.hypot(dx, dy)
            if new_cost < cost_so_far.get(nxt, float('inf')):
                cost_so_far[nxt] = new_cost
                came_from[nxt] = current
                heapq.heappush(queue, (new_cost + h(nxt), new_cost, nxt))
    if target not in came_from:
        return []
    path = []
    cell = target
    while cell is not None:
        path.append(cell)
        cell = came_from[cell]
    return path[::-1]


def expert_transitions(env, path):
    action_lookup = {tuple(map(int, action)): idx for idx,
                     action in enumerate(env.ACTIONS)}
    transitions = []
    state = env.reset()
    for current, nxt in zip(path[:-1], path[1:]):
        action = action_lookup[(nxt[0] - current[0], nxt[1] - current[1])]
        next_state, reward, done, _ = env.step(action)
        transitions.append((state, action, reward, next_state, done))
        state = next_state
        if done:
            break
    return transitions


def prefill_replay(env, agent, path, repeats=50):
    transitions = expert_transitions(env, path)
    for _ in range(repeats):
        for transition in transitions:
            agent.memory.push(*transition)


def behavior_clone(env, agent, path, epochs=120):
    transitions = expert_transitions(env, path)
    if not transitions:
        return None
    images = torch.tensor(np.stack(
        [t[0]['image'] for t in transitions]), dtype=torch.float32, device=DEVICE)
    vectors = torch.tensor(np.stack(
        [t[0]['vector'] for t in transitions]), dtype=torch.float32, device=DEVICE)
    actions = torch.tensor([t[1] for t in transitions],
                           dtype=torch.long, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    agent.current_net.train()
    for _ in range(epochs):
        logits = agent.current_net(images, vectors)
        loss = loss_fn(logits, actions)
        agent.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            agent.current_net.parameters(), agent.cfg.grad_clip)
        agent.optimizer.step()
    agent.target_net.load_state_dict(agent.current_net.state_dict())
    return float(loss.item())


In [ ]:

def train_agent(env, agent, episodes=None, verbose_every=100):
    episodes = episodes or env.cfg.episodes
    history = {'reward': [], 'steps': [],
               'success': [], 'loss': [], 'final_energy': []}
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0.0
        losses = []
        final_info = {}
        for _ in range(env.cfg.max_steps):
            action = agent.act(state, episode)
            next_state, reward, done, info = env.step(action)
            agent.memory.push(state, action, reward, next_state, done)
            loss = agent.learn()
            if loss is not None:
                losses.append(loss)
            if agent.algorithm in {'CNN-IDDQN', 'CNN-A-IDDQN'}:
                agent.update_target(soft=True)
            state = next_state
            total_reward += reward
            final_info = info
            if done:
                break
        if agent.algorithm not in {'CNN-IDDQN', 'CNN-A-IDDQN'} and episode % env.cfg.target_update == 0:
            agent.update_target(soft=False)
        history['reward'].append(total_reward)
        history['steps'].append(env.steps)
        history['success'].append(bool(final_info.get('reached', False)))
        history['loss'].append(float(np.mean(losses)) if losses else np.nan)
        history['final_energy'].append(
            float(final_info.get('energy', env.energy)))
        if verbose_every and (episode + 1) % verbose_every == 0:
            print(
                f'{agent.algorithm:15s} | episode {episode + 1:4d}/{episodes} | reward {total_reward:8.2f} | energy {env.energy:6.2f} | recent success {np.mean(history["success"][-verbose_every:]):.2f}')
    return history

In [ ]:
def greedy_rollout(env, agent):
    start_time = time.perf_counter()
    state = env.reset()
    total_reward = 0.0
    final_info = {}
    attention_trace = []
    for _ in range(env.cfg.max_steps):
        action = agent.act(state, greedy=True)
        if getattr(agent.current_net, 'last_attention', None) is not None:
            attention_trace.append(
                agent.current_net.last_attention.squeeze().numpy().tolist())
        state, reward, done, info = env.step(action)
        total_reward += reward
        final_info = info
        if done:
            break
    elapsed_ms = (time.perf_counter() - start_time) * 1000
    return list(env.path), list(env.energy_history), total_reward, final_info, elapsed_ms, attention_trace


def path_length(path):
    return float(sum(math.hypot(b[0] - a[0], b[1] - a[1]) for a, b in zip(path[:-1], path[1:])))


def count_corners(path):
    if len(path) < 3:
        return 0
    corners = 0
    prev = (path[1][0] - path[0][0], path[1][1] - path[0][1])
    for a, b in zip(path[1:-1], path[2:]):
        step = (b[0] - a[0], b[1] - a[1])
        if step != prev:
            corners += 1
        prev = step
    return corners


ASSET_IMAGES = {
    'tree': plt.imread(TREE_IMAGE_PATH),
    'apartment': plt.imread(APARTMENT_IMAGE_PATH),
    'drone': plt.imread(DRONE_IMAGE_PATH),
    'target': plt.imread(TARGET_IMAGE_PATH),
}


def add_icon(ax, image, x, y, zoom=0.060, zorder=4):
    icon = OffsetImage(image, zoom=zoom)
    box = AnnotationBbox(icon, (x, y), frameon=False, pad=0.0, zorder=zorder)
    ax.add_artist(box)


def plot_obstacles(ax, use_graphics=True):
    ax.set_facecolor('white')
    ax.figure.patch.set_facecolor('white')
    if use_graphics:
        for x, y in sorted(APARTMENT_CELLS):
            add_icon(ax, ASSET_IMAGES['apartment'], x, y, zoom=0.050, zorder=3)
        for x, y in sorted(TREE_CELLS):
            add_icon(ax, ASSET_IMAGES['tree'], x, y, zoom=0.052, zorder=4)
    else:
        for x, y in OBSTACLES:
            color = '#2f6b35' if OBSTACLE_TYPES.get((x, y)) == 'tree' else '#555555'
            ax.add_patch(plt.Rectangle((x - 0.5, y - 0.5), 1, 1, color=color))
    ax.set_xlim(-0.5, cfg.grid_size - 0.5)
    ax.set_ylim(-0.5, cfg.grid_size - 0.5)
    ax.set_aspect('equal')
    ax.set_xticks(np.arange(0, cfg.grid_size + 1, 2))
    ax.set_yticks(np.arange(0, cfg.grid_size + 1, 2))
    ax.set_xticks(np.arange(0, cfg.grid_size, 1), minor=True)
    ax.set_yticks(np.arange(0, cfg.grid_size, 1), minor=True)
    ax.grid(which='minor', color='#555555', linewidth=0.8, alpha=0.55)
    ax.grid(which='major', color='#555555', linewidth=0.8, alpha=0.55)
    ax.set_xlabel('x')
    ax.set_ylabel('y')


In [ ]:

# Full comparison option:
# ALGORITHMS = ['CNN-DQN', 'CNN-Double DQN', 'CNN-Dueling DQN', 'CNN-IDDQN', 'CNN-A-IDDQN']
# Default keeps the 30x30 graphics experiment practical to run.
ALGORITHMS = ['CNN-A-IDDQN']
RUN_TRAINING = True
USE_EXPERT_WARMSTART = True
BC_EPOCHS = {'CNN-DQN': 60, 'CNN-Double DQN': 60,
             'CNN-Dueling DQN': 60, 'CNN-IDDQN': 120, 'CNN-A-IDDQN': 140}

expert_path = astar_path()
histories, agents, rollouts, rows = {}, {}, {}, []
env = CNNCameraSensorFusionUAVGridEnv(cfg, OBSTACLES)
for algorithm in ALGORITHMS:
    print(f'Training {algorithm}...')
    agent = RLAgent(env, cfg, algorithm)
    if USE_EXPERT_WARMSTART and expert_path:
        prefill_replay(env, agent, expert_path, repeats=50)
        behavior_clone(env, agent, expert_path,
                       epochs=BC_EPOCHS.get(algorithm, 60))
    checkpoint_path = CHECKPOINT_DIR / \
        f'{algorithm.lower().replace(" ", "_")}.pth'
    if RUN_TRAINING:
        histories[algorithm] = train_agent(env, agent, verbose_every=100)
        torch.save(agent.current_net.state_dict(), checkpoint_path)
    elif checkpoint_path.exists():
        agent.current_net.load_state_dict(
            torch.load(checkpoint_path, map_location=DEVICE))
        agent.target_net.load_state_dict(agent.current_net.state_dict())
        histories[algorithm] = {'reward': [], 'steps': [],
                                'success': [], 'loss': [], 'final_energy': []}
    else:
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')
    path, energy_trace, total_reward, info, elapsed_ms, attention_trace = greedy_rollout(
        env, agent)
    agents[algorithm] = agent
    rollouts[algorithm] = {'path': path, 'energy': energy_trace, 'reward': total_reward,
                           'info': info, 'elapsed_ms': elapsed_ms, 'attention': attention_trace}
    rows.append({'algorithm': algorithm, 'reward': total_reward, 'steps': len(path) - 1, 'path_length': path_length(path), 'corners': count_corners(path), 'success': bool(info.get('reached', False)), 'final_energy': float(info.get('energy', env.energy)), 'energy_used': float(
        cfg.initial_energy - info.get('energy', env.energy)), 'inference_time_ms': elapsed_ms, 'collision': bool(info.get('collision', False)), 'out_of_bounds': bool(info.get('out_of_bounds', False)), 'energy_depleted': bool(info.get('energy_depleted', False))})

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(CSV_PATH, index=False)
reward_rows = []
for algorithm, history in histories.items():
    for episode, reward in enumerate(history['reward'], start=1):
        reward_rows.append(
            {'algorithm': algorithm, 'episode': episode, 'reward': reward})
pd.DataFrame(reward_rows).to_csv(EPISODE_REWARD_CSV_PATH, index=False)
comparison_df


In [ ]:

plt.figure(figsize=(10, 10))
ax = plt.gca()
plot_obstacles(ax)
colors = {'CNN-DQN': 'orange', 'CNN-Double DQN': 'dodgerblue',
          'CNN-Dueling DQN': 'yellowgreen', 'CNN-IDDQN': 'crimson', 'CNN-A-IDDQN': 'gold'}
styles = {'CNN-DQN': ':', 'CNN-Double DQN': ':',
          'CNN-Dueling DQN': '-.', 'CNN-IDDQN': '-', 'CNN-A-IDDQN': '--'}
for algorithm, rollout in rollouts.items():
    path = np.array(rollout['path'])
    plt.plot(path[:, 0], path[:, 1], styles[algorithm],
             color=colors[algorithm], linewidth=2.5, label=algorithm)
add_icon(ax, ASSET_IMAGES['drone'], cfg.start[0], cfg.start[1], zoom=0.075, zorder=8)
plt.text(cfg.start[0] - 1.8, cfg.start[1] - 1.1, 'SP', color='navy', fontsize=12)
add_icon(ax, ASSET_IMAGES['target'], cfg.target[0], cfg.target[1], zoom=0.075, zorder=8)
plt.text(cfg.target[0] + 0.7, cfg.target[1] - 0.8, 'TP', fontsize=12)
plt.title('30x30 CNN Camera + LiDAR + IMU Sensor-Fusion RL with Graphics Obstacles')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=200)
plt.show()

reward_df = pd.DataFrame(reward_rows)
if not reward_df.empty:
    plt.figure(figsize=(11, 5))
    for algorithm in ALGORITHMS:
        sub = reward_df[reward_df['algorithm'] == algorithm].copy()
        sub['moving_avg'] = sub['reward'].rolling(25, min_periods=1).mean()
        plt.plot(sub['episode'], sub['moving_avg'],
                 label=algorithm, linewidth=2)
    plt.xlabel('Episode')
    plt.ylabel('Reward moving average')
    plt.title('CNN Sensor-Fusion Training Reward Curves')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(REWARD_PLOT_PATH, dpi=200)
    plt.show()

metrics = ['reward', 'steps', 'path_length',
           'corners', 'final_energy', 'inference_time_ms']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, metric in zip(axes.ravel(), metrics):
    ax.bar(comparison_df['algorithm'], comparison_df[metric], color=[
           colors[a] for a in comparison_df['algorithm']])
    ax.set_title(metric)
    ax.tick_params(axis='x', labelrotation=35)
    ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(BAR_PATH, dpi=200)
plt.show()

attn = rollouts.get('CNN-A-IDDQN', {}).get('attention', [])
if attn:
    attn_arr = np.array(attn, dtype=np.float32)
    if attn_arr.ndim == 3:
        attn_arr = attn_arr[:, 0, :]
    plt.figure(figsize=(9, 4))
    for i, name in enumerate(['LiDAR', 'CNN-camera', 'IMU']):
        plt.plot(attn_arr[:, i], label=name, linewidth=2)
    plt.ylim(0, 1)
    plt.xlabel('Greedy rollout step')
    plt.ylabel('Attention weight')
    plt.title('CNN-A-IDDQN Goal-Conditioned Sensor Attention')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(ATTENTION_PLOT_PATH, dpi=200)
    plt.show()
    